# LeetCode 1185 — Day of the Week bằng Python

**Link đề:** https://leetcode.com/problems/day-of-the-week/
**Độ khó:** Easy · **Chủ đề:** Math

```python
class Solution:
    def dayOfTheWeek(self, day: int, month: int, year: int) -> str:
        ...
```

## Mục tiêu học
1. Dùng **mốc tham chiếu** (1/1/1971 là **thứ Sáu**) và **đếm số ngày** từ mốc tới ngày cần tìm.
2. Ôn lại **quy tắc năm nhuận** Gregorian và cách cộng số ngày theo năm, theo tháng.
3. Dùng phép **`% 7`** để quy số ngày về thứ trong tuần; hiểu vì sao phải **dịch chỉ số** theo thứ của mốc.
4. Biết thêm công thức **Zeller** / **Sakamoto** và thư viện `datetime` để đối chiếu.
5. Trace, chứng minh, test toàn miền, lỗi hay gặp, biến thể, bài tập.

## 1. Đề bài và ví dụ

Cho một ngày gồm ba số `day`, `month`, `year`. Trả về thứ trong tuần của ngày đó, là một trong:
`{"Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"}`.

**Gợi ý của đề:** ngày 1/1/1971 là **Friday** (thứ Sáu).

| day | month | year | Output |
|---:|---:|---:|---|
| 31 | 8 | 2019 | `"Saturday"` |
| 18 | 7 | 1999 | `"Sunday"` |
| 15 | 8 | 1993 | `"Sunday"` |

**Constraints:** ngày cho vào là ngày hợp lệ trong khoảng năm **1971 – 2100**.

In [1]:
DAYS = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]
print(len(DAYS), DAYS.index("Friday"))

7 5


## 2. Nền tảng cần biết

### 2.1. Thứ trong tuần lặp lại theo chu kỳ 7

Nếu hôm nay là thứ `w` (đánh số 0..6), thì sau `k` ngày là thứ `(w + k) % 7`. Ví dụ: hôm nay Friday (5), sau 10 ngày: `(5 + 10) % 7 = 1` → Monday.

In [2]:
w = DAYS.index("Friday")
for k in [0, 1, 2, 7, 10, 365]:
    print(f"sau {k:>3} ngày: {DAYS[(w + k) % 7]}")

sau   0 ngày: Friday
sau   1 ngày: Saturday
sau   2 ngày: Sunday
sau   7 ngày: Friday
sau  10 ngày: Monday
sau 365 ngày: Saturday


Vì sao `365` ngày sau lại là Saturday? `365 = 52·7 + 1` → dịch 1 thứ. Năm nhuận `366 = 52·7 + 2` → dịch 2 thứ. Đây là lý do cùng một ngày tháng sang năm sau thường "lùi" sang thứ kế tiếp.

### 2.2. Năm nhuận Gregorian

```python
leap = (y % 4 == 0 and y % 100 != 0) or y % 400 == 0
```
Trong miền đề: **2000 nhuận**, **2100 không nhuận**.

In [3]:
def is_leap(y: int) -> bool:
    return (y % 4 == 0 and y % 100 != 0) or y % 400 == 0

print([y for y in range(1971, 1990) if is_leap(y)])
print(is_leap(2000), is_leap(2100))

[1972, 1976, 1980, 1984, 1988]
True False


### 2.3. Số ngày mỗi tháng và `sum` các tháng trước

```python
month_days = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
```
Số ngày từ 1/1 tới trước tháng `m` = `sum(month_days[:m-1])` (+1 nếu nhuận và `m > 2`).

In [4]:
month_days = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
print("trước tháng 8:", sum(month_days[:7]))

trước tháng 8: 212


### 2.4. Toán tử `%` với số âm trong Python

`(-1) % 7 = 6` — Python luôn trả số dư **không âm** khi số chia dương. Điều này giúp các công thức "lùi ngày" không cần xử lý riêng.

In [5]:
print((-1) % 7, (-8) % 7, 13 % 7)

6 6 6


## 3. Cách 1 — Trực giác: đi từng ngày từ mốc 1/1/1971

Bắt đầu ở 1/1/1971 (Friday), tăng ngày lên một, đếm số bước tới ngày cần tìm. Đúng nhưng có thể tới ~47 000 bước (1971 → 2100).

In [6]:
def day_of_week_walk(day: int, month: int, year: int) -> str:
    d, m, y = 1, 1, 1971
    steps = 0
    while (d, m, y) != (day, month, year):
        dim = month_days[m - 1] + (1 if m == 2 and is_leap(y) else 0)   # số ngày tháng m
        d += 1
        if d > dim:
            d, m = 1, m + 1
            if m > 12:
                m, y = 1, y + 1
        steps += 1
    return DAYS[(5 + steps) % 7]

print(day_of_week_walk(31, 8, 2019), day_of_week_walk(18, 7, 1999), day_of_week_walk(15, 8, 1993))

Saturday Sunday Sunday


So sánh tuple `(d, m, y) != (day, month, year)`: Python so từng phần tử — tiện để kiểm tra "đã tới ngày đích chưa".

### Nút thắt
Ta đếm **từng ngày** trong khi có thể cộng **cả năm** (365/366) và **cả tháng** một lần.

## 4. Insight

```text
số ngày từ 1/1/1971 tới ngày cần tìm =
    Σ (365 hoặc 366) cho các năm 1971 .. year-1
  + Σ số ngày các tháng 1 .. month-1 của năm year
  + (day - 1)

thứ = DAYS[(5 + số_ngày) % 7]     # 5 = vị trí "Friday" trong DAYS
```

Ví dụ 3/1/1971: số ngày = 2 → `(5 + 2) % 7 = 0` → Sunday ✔ (1/1 Friday, 2/1 Saturday, 3/1 Sunday).

In [7]:
print(DAYS[(5 + 0) % 7], DAYS[(5 + 1) % 7], DAYS[(5 + 2) % 7])

Friday Saturday Sunday


## 5. Cách 2 — Lời giải chính: đếm ngày từ mốc

**Mổ xẻ:**
1. `days = 0`.
2. Cộng số ngày của mỗi năm từ 1971 tới `year - 1`.
3. Cộng số ngày các tháng trước `month` (tháng 2 có 29 nếu `year` nhuận).
4. Cộng `day - 1` (vì 1/1/1971 tính là ngày 0).
5. Trả `DAYS[(days + 5) % 7]`.

In [8]:
def day_of_week(day: int, month: int, year: int) -> str:
    days = 0
    for y in range(1971, year):
        days += 366 if is_leap(y) else 365
    md = [31, 29 if is_leap(year) else 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    days += sum(md[:month - 1])
    days += day - 1
    return DAYS[(days + 5) % 7]

print(day_of_week(31, 8, 2019), day_of_week(18, 7, 1999), day_of_week(15, 8, 1993))

Saturday Sunday Sunday


## 6. TRACE từng bước với 31/8/2019

| phần | cách tính | giá trị |
|---|---|---:|
| các năm 1971..2018 | 48 năm, trong đó 12 năm nhuận (1972, 1976, …, 2016) | `48·365 + 12 = 17532` |
| các tháng 1..7 năm 2019 | `31+28+31+30+31+30+31` | `212` |
| `day - 1` | `31 - 1` | `30` |
| **tổng** | | `17774` |
| thứ | `(17774 + 5) % 7 = 17779 % 7 = 6` | **Saturday** |

In [9]:
def trace_day_of_week(day: int, month: int, year: int) -> str:
    years = list(range(1971, year))
    leaps = [y for y in years if is_leap(y)]
    part_years = len(years) * 365 + len(leaps)
    print(f"ngày {day}/{month}/{year}")
    print(f"  {len(years)} năm trước đó, {len(leaps)} năm nhuận → {part_years} ngày")
    md = [31, 29 if is_leap(year) else 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    part_months = sum(md[:month - 1])
    print(f"  các tháng trước: {md[:month - 1]} → {part_months} ngày")
    total = part_years + part_months + day - 1
    print(f"  + (day - 1) = {day - 1} → tổng {total} ngày kể từ 1/1/1971")
    idx = (total + 5) % 7
    print(f"  ({total} + 5) % 7 = {idx} → {DAYS[idx]}")
    return DAYS[idx]

trace_day_of_week(31, 8, 2019)
print()
trace_day_of_week(1, 3, 2000)

ngày 31/8/2019
  48 năm trước đó, 12 năm nhuận → 17532 ngày
  các tháng trước: [31, 28, 31, 30, 31, 30, 31] → 212 ngày
  + (day - 1) = 30 → tổng 17774 ngày kể từ 1/1/1971
  (17774 + 5) % 7 = 6 → Saturday

ngày 1/3/2000
  29 năm trước đó, 7 năm nhuận → 10592 ngày
  các tháng trước: [31, 29] → 60 ngày
  + (day - 1) = 0 → tổng 10652 ngày kể từ 1/1/1971
  (10652 + 5) % 7 = 3 → Wednesday


'Wednesday'

Ví dụ thứ hai đi qua tháng 2/2000 — năm nhuận đặc biệt (chia hết cho 400) — nên tháng 2 có 29 ngày.

## 7. Cách 3 — Đếm năm nhuận bằng công thức (không vòng lặp theo năm)

Số năm nhuận trong `[1, y]` là `y//4 - y//100 + y//400`. Số năm nhuận trong `[1971, year-1]` = `f(year-1) - f(1970)`.

In [10]:
def leaps_upto(y: int) -> int:
    return y // 4 - y // 100 + y // 400

def day_of_week_formula(day: int, month: int, year: int) -> str:
    n_years = year - 1971
    n_leaps = leaps_upto(year - 1) - leaps_upto(1970)
    days = n_years * 365 + n_leaps
    before = [0, 31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334]
    days += before[month - 1] + (1 if month > 2 and is_leap(year) else 0)
    days += day - 1
    return DAYS[(days + 5) % 7]

print(leaps_upto(2018) - leaps_upto(1970))   # 12 năm nhuận trong 1971..2018
print(day_of_week_formula(31, 8, 2019), day_of_week_formula(18, 7, 1999))

12
Saturday Sunday


## 8. Cách 4 — Thuật toán Sakamoto (công thức đóng)

Tomohiko Sakamoto đưa ra công thức ngắn gọn (0 = Sunday):

```python
t = [0, 3, 2, 5, 0, 3, 5, 1, 4, 6, 2, 4]
if month < 3: year -= 1
w = (year + year//4 - year//100 + year//400 + t[month-1] + day) % 7
```

Ý tưởng: coi tháng 1, 2 thuộc **năm trước** để ngày nhuận (29/2) rơi vào **cuối** "năm", khi đó số năm nhuận chỉ phụ thuộc `year`. Bảng `t` lưu độ lệch thứ của ngày đầu mỗi tháng. Học để biết — không bắt buộc nhớ.

In [11]:
def day_of_week_sakamoto(day: int, month: int, year: int) -> str:
    t = [0, 3, 2, 5, 0, 3, 5, 1, 4, 6, 2, 4]
    if month < 3:
        year -= 1
    w = (year + year // 4 - year // 100 + year // 400 + t[month - 1] + day) % 7
    return DAYS[w]

print(day_of_week_sakamoto(31, 8, 2019), day_of_week_sakamoto(18, 7, 1999), day_of_week_sakamoto(15, 8, 1993))

Saturday Sunday Sunday


## 9. Cách 5 — Thư viện `datetime` (để đối chiếu)

`datetime.date(y, m, d).weekday()` trả **0 = Monday … 6 = Sunday** (khác thứ tự của ta!). Phải đổi: chỉ số trong `DAYS` = `(weekday + 1) % 7`. Hoặc dùng `.strftime("%A")` ra tên tiếng Anh (phụ thuộc locale, thường là tiếng Anh).

In [12]:
import datetime

def day_of_week_lib(day: int, month: int, year: int) -> str:
    return DAYS[(datetime.date(year, month, day).weekday() + 1) % 7]

d = datetime.date(2019, 8, 31)
print(d.weekday(), d.isoweekday(), d.strftime("%A"))
print(day_of_week_lib(31, 8, 2019), datetime.date(1971, 1, 1).strftime("%A"))

5 6 Saturday
Saturday Friday


## 10. Code nộp LeetCode

In [13]:
class Solution:
    def dayOfTheWeek(self, day: int, month: int, year: int) -> str:
        names = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]

        def leap(y: int) -> bool:
            return (y % 4 == 0 and y % 100 != 0) or y % 400 == 0

        days = 0
        for y in range(1971, year):
            days += 366 if leap(y) else 365
        md = [31, 29 if leap(year) else 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
        days += sum(md[:month - 1]) + day - 1
        return names[(days + 5) % 7]          # 1/1/1971 là Friday (index 5)

sol = Solution()
print(sol.dayOfTheWeek(31, 8, 2019), sol.dayOfTheWeek(18, 7, 1999), sol.dayOfTheWeek(15, 8, 1993))

Saturday Sunday Sunday


In [14]:
class SolutionSakamoto:
    def dayOfTheWeek(self, day: int, month: int, year: int) -> str:
        names = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]
        t = [0, 3, 2, 5, 0, 3, 5, 1, 4, 6, 2, 4]
        if month < 3:
            year -= 1
        return names[(year + year // 4 - year // 100 + year // 400 + t[month - 1] + day) % 7]

print(SolutionSakamoto().dayOfTheWeek(31, 8, 2019))

Saturday


## 11. Vì sao đúng?

- Gọi `D` = số ngày từ 1/1/1971 tới ngày cần tìm (1/1/1971 ứng với `D = 0`). Ngày có thứ = thứ của mốc dịch đi `D` ngày, tức chỉ số `(5 + D) % 7` trong `DAYS` (vì thứ lặp chu kỳ 7 và mốc là Friday, chỉ số 5).
- `D` được tính đúng: mỗi năm trọn vẹn đóng góp 365 hoặc 366 ngày theo quy tắc Gregorian; các tháng trọn vẹn trước `month` đóng góp tổng độ dài của chúng (tháng 2 tính theo `year`); trong tháng hiện tại có `day - 1` ngày đã trôi qua.
- **Invariant** khi cộng năm: sau khi cộng năm `y`, `days` = số ngày từ 1/1/1971 tới 1/1/(y+1).

Test toàn bộ ~47 000 ngày trong miền đề bằng `datetime` (mục 13) xác nhận thêm bằng thực nghiệm.

## 12. Độ phức tạp

Gọi `Y = year - 1971` (≤ 129).

| Cách | Time | Space |
|---|---|---|
| Đi từng ngày | `O(Y·365)` | `O(1)` |
| Đếm ngày theo năm/tháng | `O(Y + 12)` | `O(1)` |
| Công thức năm nhuận | `O(1)` | `O(1)` |
| Sakamoto | `O(1)` | `O(1)` |

Vì `Y` bị chặn bởi 129, mọi cách đều nhanh; nhưng công thức `O(1)` vẫn đúng với năm bất kỳ.

## 13. Bộ test: mẫu, biên, toàn miền

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<22} PASS ({actual})")

sol = Solution()
check("đề 1", sol.dayOfTheWeek(31, 8, 2019), "Saturday")
check("đề 2", sol.dayOfTheWeek(18, 7, 1999), "Sunday")
check("đề 3", sol.dayOfTheWeek(15, 8, 1993), "Sunday")
check("mốc 1/1/1971", sol.dayOfTheWeek(1, 1, 1971), "Friday")
check("29/2/2000 (nhuận)", sol.dayOfTheWeek(29, 2, 2000), "Tuesday")
check("1/3/2100 (không nhuận)", sol.dayOfTheWeek(1, 3, 2100), "Monday")
check("31/12/2100", sol.dayOfTheWeek(31, 12, 2100), "Friday")

đề 1                   PASS (Saturday)
đề 2                   PASS (Sunday)
đề 3                   PASS (Sunday)
mốc 1/1/1971           PASS (Friday)
29/2/2000 (nhuận)      PASS (Tuesday)
1/3/2100 (không nhuận) PASS (Monday)
31/12/2100             PASS (Friday)


In [16]:
solvers = [day_of_week, day_of_week_formula, day_of_week_sakamoto,
           Solution().dayOfTheWeek, SolutionSakamoto().dayOfTheWeek]
cur = datetime.date(1971, 1, 1)
end = datetime.date(2100, 12, 31)
count = 0
while cur <= end:
    expected = day_of_week_lib(cur.day, cur.month, cur.year)
    for f in solvers:
        assert f(cur.day, cur.month, cur.year) == expected, (f, cur)
    cur += datetime.timedelta(days=1)
    count += 1
print(f"Đã kiểm tra {count} ngày (1971–2100) với 5 lời giải: khớp datetime. PASS")

for args in [(1, 1, 1971), (31, 12, 1975), (15, 8, 1993)]:
    assert day_of_week_walk(*args) == day_of_week(*args)
print("Bản đi từng ngày khớp trên vài ngày mẫu. PASS")

Đã kiểm tra 47482 ngày (1971–2100) với 5 lời giải: khớp datetime. PASS
Bản đi từng ngày khớp trên vài ngày mẫu. PASS


## 14. Lỗi người mới hay gặp

### Lỗi 1: Quên dịch theo thứ của mốc
`DAYS[days % 7]` coi 1/1/1971 là Sunday → lệch 5 ngày.

### Lỗi 2: Cộng `day` thay vì `day - 1` → lệch 1 ngày (mốc phải ứng với 0).

### Lỗi 3: Kiểm tra nhuận của **năm hiện tại** nhưng cho tháng 1–2
Ngày 29/2 chỉ ảnh hưởng các tháng **sau** tháng 2. Với bản "cộng `before[month-1]`", chỉ cộng thêm 1 khi `month > 2`.

### Lỗi 4: Dùng `weekday()` của `datetime` như thể 0 là Sunday.

In [17]:
def wrong_no_offset(day, month, year):
    days = sum(366 if is_leap(y) else 365 for y in range(1971, year))
    md = [31, 29 if is_leap(year) else 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    days += sum(md[:month - 1]) + day - 1
    return DAYS[days % 7]

def wrong_day_not_minus_one(day, month, year):
    days = sum(366 if is_leap(y) else 365 for y in range(1971, year))
    md = [31, 29 if is_leap(year) else 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    days += sum(md[:month - 1]) + day
    return DAYS[(days + 5) % 7]

def wrong_leap_jan(day, month, year):
    before = [0, 31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334]
    days = sum(366 if is_leap(y) else 365 for y in range(1971, year))
    days += before[month - 1] + (1 if is_leap(year) else 0) + day - 1    # thiếu điều kiện month > 2
    return DAYS[(days + 5) % 7]

def wrong_weekday(day, month, year):
    return DAYS[datetime.date(year, month, day).weekday()]

args = (31, 8, 2019)
print("đúng:", day_of_week(*args))
print("quên dịch mốc:", wrong_no_offset(*args))
print("cộng day:", wrong_day_not_minus_one(*args))
print("weekday() sai gốc:", wrong_weekday(*args))
print("15/1/2000 đúng:", day_of_week(15, 1, 2000), "| cộng nhuận cho tháng 1:", wrong_leap_jan(15, 1, 2000))

đúng: Saturday
quên dịch mốc: Monday
cộng day: Sunday
weekday() sai gốc: Friday
15/1/2000 đúng: Saturday | cộng nhuận cho tháng 1: Sunday


## 15. Mẹo nhận dạng pattern

- "Ngày X là thứ mấy, biết một mốc" → **đếm số ngày chênh lệch rồi `% 7`**.
- Bài ngày tháng = **phân rã**: năm trọn → tháng trọn → ngày lẻ; luôn hỏi "năm nhuận?" và "đếm từ 0 hay 1?".
- Muốn `O(1)` theo năm: đếm năm nhuận bằng `y//4 - y//100 + y//400`.
- Kiểm thử: `datetime` của thư viện chuẩn là "oracle" tin cậy cho mọi bài lịch Gregorian.

Bài cùng họ: 1154 (Day of the Year), 1360 (Number of Days Between Two Dates), 1507 (Reformat Date).

## 16. Biến thể

### 16.1. Số ngày giữa hai ngày (1360)
Quy cả hai về "số ngày từ mốc" rồi trừ.

In [18]:
def days_from_1971(day: int, month: int, year: int) -> int:
    days = sum(366 if is_leap(y) else 365 for y in range(1971, year))
    md = [31, 29 if is_leap(year) else 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    return days + sum(md[:month - 1]) + day - 1

def days_between(d1: str, d2: str) -> int:
    y1, m1, a1 = map(int, d1.split("-"))
    y2, m2, a2 = map(int, d2.split("-"))
    return abs(days_from_1971(a1, m1, y1) - days_from_1971(a2, m2, y2))

print(days_between("2019-06-29", "2019-06-30"), days_between("2020-01-15", "2019-12-31"))

1 15


### 16.2. Thứ của ngày 13 mỗi tháng — đếm "Thứ Sáu ngày 13"

In [19]:
def friday_13_in_year(year: int) -> list:
    return [m for m in range(1, 13) if day_of_week(13, m, year) == "Friday"]

for y in [2015, 2019, 2026]:
    print(y, friday_13_in_year(y))

2015 [2, 3, 11]
2019 [9, 12]
2026 [2, 3, 11]


### 16.3. So sánh các cách

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| Đi từng ngày | mô phỏng lịch | `O(Y·365)` | `O(1)` | dễ nghĩ, chậm |
| Đếm theo năm/tháng | cộng khối + `% 7` | `O(Y)` | `O(1)` | **nộp bài**, dễ giải thích |
| Công thức năm nhuận | `y//4 - y//100 + y//400` | `O(1)` | `O(1)` | tổng quát |
| Sakamoto | công thức đóng | `O(1)` | `O(1)` | ngắn, khó tự nghĩ ra |
| `datetime` | thư viện | `O(1)` | `O(1)` | oracle kiểm thử |

## 17. Bài tập vận dụng (TODO)

### Bài 1 — Tính tay
1/1/1972 là thứ mấy? 1/1/1973 là thứ mấy? (Gợi ý: 1971 có 365 ngày, 1972 nhuận.)

### Bài 2 — Đếm số năm nhuận trong một khoảng

In [20]:
def count_leaps_exercise(a: int, b: int) -> int:
    # TODO: số năm nhuận trong [a, b], có thể dùng vòng lặp hoặc công thức
    pass

# assert count_leaps_exercise(1971, 2018) == 12
# assert count_leaps_exercise(1900, 1900) == 0
# assert count_leaps_exercise(2000, 2000) == 1

### Bài 3 — Hoàn thiện lời giải chính

In [21]:
def day_of_week_exercise(day: int, month: int, year: int) -> str:
    # TODO 1: cộng số ngày các năm 1971..year-1
    # TODO 2: cộng các tháng trước (tháng 2 nhuận?)
    # TODO 3: cộng day - 1, rồi trả DAYS[(days + 5) % 7]
    pass

# assert day_of_week_exercise(31, 8, 2019) == "Saturday"
# assert day_of_week_exercise(18, 7, 1999) == "Sunday"

### Bài 4 — Đổi mốc
Viết lại lời giải chính nhưng dùng mốc **1/1/2000 là Saturday** (chỉ cho năm ≥ 2000).

In [22]:
def day_of_week_2000_exercise(day: int, month: int, year: int) -> str:
    # TODO: chỉ số của "Saturday" trong DAYS là 6
    pass

# assert day_of_week_2000_exercise(31, 8, 2019) == "Saturday"
# assert day_of_week_2000_exercise(1, 1, 2000) == "Saturday"

### Bài 5 — Ngày trong tuần sau `k` ngày
Cho tên thứ hiện tại và số nguyên `k` (có thể âm), trả tên thứ sau `k` ngày.

In [23]:
def shift_weekday_exercise(name: str, k: int) -> str:
    # TODO: DAYS.index + k, rồi % 7
    pass

# assert shift_weekday_exercise("Friday", 2) == "Sunday"
# assert shift_weekday_exercise("Monday", -1) == "Sunday"
# assert shift_weekday_exercise("Sunday", 700) == "Sunday"

### Bài 6 — Mini challenge: đếm số Chủ nhật rơi vào ngày 1 của tháng
Trong khoảng năm `[a, b]` (1971 ≤ a ≤ b ≤ 2100), có bao nhiêu tháng có ngày 1 là Sunday? (Biến thể của Project Euler #19.)

In [24]:
def count_first_sundays_exercise(a: int, b: int) -> int:
    # TODO
    pass

# assert count_first_sundays_exercise(2019, 2019) == 2      # tháng 9 và tháng 12

## 18. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | 365 = 52·7 + 1 | Friday + 1 | 1972 nhuận → +2 |
| 2 | dùng `is_leap` | `sum(... for y in range(a, b + 1))` | hoặc `f(b) - f(a - 1)` |
| 3 | xem mục 5 | `md[:month - 1]` | `DAYS[(days + 5) % 7]` |
| 4 | đổi 1971 → 2000 | đổi 5 → 6 | `range(2000, year)` |
| 5 | chu kỳ 7 | `(i + k) % 7` | `%` của Python xử lý `k` âm |
| 6 | duyệt mọi (năm, tháng) | gọi `day_of_week(1, m, y)` | đếm `== "Sunday"` |

## 19. Đáp án tham khảo

**Bài 1:** 1/1/1972 = Friday + 365 ngày = Friday + 1 = **Saturday**. 1/1/1973 = Saturday + 366 ngày = Saturday + 2 = **Monday**.

In [25]:
def count_leaps_answer(a: int, b: int) -> int:
    return leaps_upto(b) - leaps_upto(a - 1)

def day_of_week_answer(day: int, month: int, year: int) -> str:
    days = 0
    for y in range(1971, year):
        days += 366 if is_leap(y) else 365
    md = [31, 29 if is_leap(year) else 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    days += sum(md[:month - 1]) + day - 1
    return DAYS[(days + 5) % 7]

def day_of_week_2000_answer(day: int, month: int, year: int) -> str:
    days = 0
    for y in range(2000, year):
        days += 366 if is_leap(y) else 365
    md = [31, 29 if is_leap(year) else 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    days += sum(md[:month - 1]) + day - 1
    return DAYS[(days + 6) % 7]

def shift_weekday_answer(name: str, k: int) -> str:
    return DAYS[(DAYS.index(name) + k) % 7]

def count_first_sundays_answer(a: int, b: int) -> int:
    return sum(1 for y in range(a, b + 1) for m in range(1, 13) if day_of_week(1, m, y) == "Sunday")

assert day_of_week(1, 1, 1972) == "Saturday" and day_of_week(1, 1, 1973) == "Monday"
assert count_leaps_answer(1971, 2018) == 12
assert count_leaps_answer(1900, 1900) == 0
assert count_leaps_answer(2000, 2000) == 1
assert day_of_week_answer(31, 8, 2019) == "Saturday"
assert day_of_week_answer(18, 7, 1999) == "Sunday"
assert day_of_week_2000_answer(31, 8, 2019) == "Saturday"
assert day_of_week_2000_answer(1, 1, 2000) == "Saturday"
assert shift_weekday_answer("Friday", 2) == "Sunday"
assert shift_weekday_answer("Monday", -1) == "Sunday"
assert shift_weekday_answer("Sunday", 700) == "Sunday"
assert count_first_sundays_answer(2019, 2019) == 2
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 20. Template nhớ nhanh + checklist submit

```python
class Solution:
    def dayOfTheWeek(self, day: int, month: int, year: int) -> str:
        names = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]
        leap = lambda y: (y % 4 == 0 and y % 100 != 0) or y % 400 == 0
        days = sum(366 if leap(y) else 365 for y in range(1971, year))
        md = [31, 29 if leap(year) else 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
        days += sum(md[:month - 1]) + day - 1
        return names[(days + 5) % 7]
```

**Checklist:**
- [ ] Mốc 1/1/1971 ứng với **0 ngày** và chỉ số **5** (Friday) trong list bắt đầu từ Sunday?
- [ ] Cộng `day - 1`, không phải `day`?
- [ ] Quy tắc nhuận đủ 3 điều kiện (2100 không nhuận)?
- [ ] Tháng 2 năm nhuận chỉ ảnh hưởng các tháng **sau** tháng 2?
- [ ] Test mốc, 29/2/2000, 1/3/2100 và ba ví dụ của đề?